# Huấn luyện Distill Kokoro-7M Tiếng Việt (Dataset: ngochuyen001)

Notebook này thực hiện chưng cất và căn chỉnh mô hình **Kokoro-7M** (7.48M tham số, nhẹ hơn 11 lần so với bản 82M) cho giọng đọc tiếng Việt của Ngọc Huyền.

### Yêu cầu cấu hình Kaggle:
- **Accelerator**: GPU Tesla T4 x 1 hoặc P100 (16GB VRAM)
- **Internet**: **ON**
- **Persistence**: Filesystem variables
- **Dataset Input**: `/kaggle/input/datasets/xkorenguyenthinh/ngochuyen001/content/audio`

In [ ]:
# 1. Cài đặt hệ thống và các thư viện cần thiết
!apt-get update -qq && apt-get install -y -qq ffmpeg libsndfile1
!pip install -q soundfile soxr librosa transformers torchaudio vig2p

import torch
print(f"CUDA Available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"Device: {torch.cuda.get_device_name(0)}")
    print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.2f} GB")

In [ ]:
# 2. Thiết lập thư mục làm việc và mã nguồn Kokoro-7M-NgocHuyen
import os, sys

# Kiểm tra nếu đang chạy trong repo đã copy hoặc đang ở /kaggle/working
WORKING_DIR = "/kaggle/working/Kokoro-7M-NgocHuyen"
if not os.path.exists(WORKING_DIR):
    # Thử tìm kiếm trong input hoặc tạo thư mục làm việc trực tiếp
    LOCAL_SRC = "/kaggle/input/kokoro-7m-ngochuyen"
    if os.path.exists(LOCAL_SRC):
        !cp -r /kaggle/input/kokoro-7m-ngochuyen /kaggle/working/Kokoro-7M-NgocHuyen
    else:
        os.makedirs(WORKING_DIR, exist_ok=True)

os.chdir(WORKING_DIR if os.path.exists(WORKING_DIR) else "/kaggle/working")
print(f"Current working directory: {os.getcwd()}")

if os.getcwd() not in sys.path:
    sys.path.insert(0, os.getcwd())
patched_path = os.path.join(os.getcwd(), "kokoro_patched")
if os.path.exists(patched_path) and patched_path not in sys.path:
    sys.path.insert(0, patched_path)

In [ ]:
# 3. Tải base weights để warm-start (kokoro_en_7m.pth) & model trích xuất style
!wget -nc -q https://huggingface.co/oddadmix/Kokoro-7M-Distill/resolve/main/kokoro_en_7m.pth -O kokoro_en_7m.pth
!wget -nc -q https://huggingface.co/hexgrad/Kokoro-82M/resolve/main/kokoro-v0_19.pth -O kokoro-v0_19.pth

print("Downloaded base checkpoints:")
!ls -lh *.pth

## Giai đoạn 1: Tiền xử lý Dữ liệu & Trích xuất Voicepack

In [ ]:
# 4. Tiền xử lý audio ngochuyen001 sang 24kHz FLAC và tạo vi_manifest.jsonl
DATA_DIR = "/kaggle/input/datasets/xkorenguyenthinh/ngochuyen001/content/audio"
META_FILE = os.path.join(DATA_DIR, "metadata.csv")

!python prep_manifest_vi.py \
    --data-dir "$DATA_DIR" \
    --metadata "$META_FILE" \
    --out-dir data/wavs \
    --out-manifest data/vi_manifest.jsonl

# Kiểm tra 3 dòng manifest đầu tiên
!head -n 3 data/vi_manifest.jsonl

In [ ]:
# 5. Trích xuất Voicepack style vector (vi_voice.pt)
!python extract_voicepack.py \
    --model kokoro-v0_19.pth \
    --audio-dir data/wavs \
    --output vi_voice.pt \
    --num-samples 200

import torch
vp = torch.load("vi_voice.pt", map_location="cpu")
print(f"Voicepack loaded successfully! Shape: {vp.shape}")

## Giai đoạn 2: Huấn luyện BiLSTM CTC Aligner (40Hz Grid)

In [ ]:
# 6. Huấn luyện CTC aligner trực tiếp trên giọng Ngọc Huyền
!python train_aligner.py \
    --manifest data/vi_manifest.jsonl \
    --max-frames 1000 \
    --steps 24000 \
    --batch 16 \
    --out runs/aligner_vi \
    --resume

## Giai đoạn 3: Forced Alignment & Gate Kiểm Định (Bắt buộc)

In [ ]:
# 7. Chạy Probe 500 clips để đo Frame Shift tối ưu
!python align_ctc.py \
    --ckpt runs/aligner_vi/last.pt \
    --manifest data/vi_manifest.jsonl \
    --max-frames 1000 \
    --limit 500 \
    --out data/index_probe.jsonl

# Tính toán SHIFT tối ưu
import subprocess
res = subprocess.run(["python", "pick_shift.py", "data/index_probe.jsonl", "config_student_wide.json"], capture_output=True, text=True)
SHIFT = res.stdout.strip().splitlines()[-1]
print(f"Measured optimal shift: {SHIFT}")
print(res.stderr)

In [ ]:
# 8. Forced alignment toàn bộ dataset với Frame Shift đã đo
!python align_ctc.py \
    --ckpt runs/aligner_vi/last.pt \
    --manifest data/vi_manifest.jsonl \
    --max-frames 1000 \
    --shift "$SHIFT" \
    --seg-max 800 \
    --seg-min 60 \
    --out data/index_raw.jsonl

# Cắt đoạn tránh tràn ALBERT 512 tokens limit
!python resegment.py data/index_raw.jsonl data/index_ctc.jsonl

# Kiểm định Gate 16 (Quality Gate)
!python gate16.py data/index_ctc.jsonl config_student_wide.json 600

## Giai đoạn 4: Distill / Finetune Kokoro-7M Tiếng Việt

In [ ]:
# 9. Huấn luyện Student 7M
!python train_student.py \
    --steps 40000 \
    --batch 8 \
    --lr 5e-5 \
    --mel-weight 5.0 \
    --sil-weight 100.0 \
    --slm-weight 1.0 \
    --slm-adv-weight 0.2 \
    --init-from kokoro_en_7m.pth \
    --voicepack vi_voice.pt \
    --index data/index_ctc.jsonl \
    --config config_student_wide.json \
    --out runs/kokoro_vi_7m \
    --resume \
    --ckpt-every 2000

## Giai đoạn 5: Đóng gói Checkpoint & Kiểm thử Inference

In [ ]:
# 10. Chuyển đổi checkpoint sang định dạng lồng (Nested) cho KModel
!python export_model.py \
    --src runs/kokoro_vi_7m/student.pth \
    --dst kokoro_vi_7m.pth

In [ ]:
# 11. Thử nghiệm phát âm thanh tiếng Việt
from IPython.display import Audio, display

sample_text = "Xin chào quý vị và các bạn! Đây là giọng đọc tiếng Việt được chưng cất thành công vào mô hình cô cô rô bảy mê ga byte."

!python infer_test.py \
    --text "$sample_text" \
    --model kokoro_vi_7m.pth \
    --voicepack vi_voice.pt \
    --out test_sample.wav

display(Audio("test_sample.wav", rate=24000))

In [ ]:
# 12. Nén các file kết quả để tải về hoặc lưu trữ vào Kaggle Output
!zip -r kokoro_vi_7m_release.zip \
    kokoro_vi_7m.pth \
    vi_voice.pt \
    config_student_wide.json \
    test_sample.wav \
    kokoro_symbols.py

print("Tập tin nén sẵn sàng để download:")
!ls -lh kokoro_vi_7m_release.zip